## Parking Lot Design
### Problem Statement
Design a multi-floor parking lot system that can manage the entry and exit of vehicles, assign appropriate parking spots based on vehicle type, issue timestamped tickets, calculate fees, and display real-time availability.

The system must handle multiple vehicle types across multiple floors, enforce business rules around spot sizing, support pluggable fee strategies, and be extensible without modifying core logic.

There are three sizes of parking spot - small, medium and large allowing three vehicle sizes (motorcycles, cars and trucks). Smaller vehicle can occupy larger parking spot. 

### Solution
We define the following classes:

**`Vehicle`:** stores information about the vehicle including its size (in `VehicleType` enum)

In [ ]:
@Getter
@Setter
@AllArgsConstructor
public class Vehicle {
    private String registrationNumber;
    private VehicleType vehicleType;
}

// Each enum instance also stores which ParkingSpot size it is compatible with
public enum VehicleType {
    MOTORCYCLE {
        @Override
        public Set<ParkingSpotSize> compatibleSpots() {
            return Set.of(ParkingSpotSize.SMALL, ParkingSpotSize.MEDIUM, ParkingSpotSize.LARGE);
        }
    },
    CAR {
        @Override
        public Set<ParkingSpotSize> compatibleSpots() {
            return Set.of(ParkingSpotSize.MEDIUM, ParkingSpotSize.LARGE);
        }
    },
    BUS {
        @Override
        public Set<ParkingSpotSize> compatibleSpots() {
            return Set.of(ParkingSpotSize.LARGE);
        }
    };

    public abstract Set<ParkingSpotSize> compatibleSpots();
}

**`ParkingTicket`:** that contains information about parking status, location, fees, etc.

In [ ]:
@Getter
@Setter
public class ParkingTicket {
    private String id;
    private LocalDateTime startTime;
    private LocalDateTime endTime;
    private ParkingSpot parkingSpot;
    private Vehicle vehicle;
    private BigDecimal fees;
    private TicketStatus status;

    public ParkingTicket() {
        this.setId(UUID.randomUUID().toString());
        this.setStartTime(LocalDateTime.now());
        this.status = TicketStatus.ACTIVE;
    }
}

public enum TicketStatus {
    ACTIVE,
    PAID,
}

**`ParkingSpot`:** representing one parking spot. `ParkingSpotSize` enum represents sizes.

In [ ]:
@Getter
@Setter
public class ParkingSpot {
    private String id;
    private int floor;
    private ParkingSpotSize size;
    private boolean occupied;
}

public enum ParkingSpotSize {
    SMALL,
    MEDIUM,
    LARGE
}

**`ParkingFloor`:** containing multiple parking spots. Thread safe implementation to get and release spots.

In [ ]:
@Getter
@Setter
public class ParkingFloor {
    private int floor;
    private List<ParkingSpot> parkingSpots;
    private final ReentrantLock lock = new ReentrantLock();

    public ParkingSpot getParkingSpot(Set<ParkingSpotSize> parkingSpotSizes) {
        lock.lock();
        try {
            for (ParkingSpot spot : parkingSpots) {
                if (!spot.isOccupied() && parkingSpotSizes.contains(spot.getSize())) {
                    spot.setOccupied(true);
                    return spot;
                }
            }

            return null;
        } finally {
            lock.unlock();
        }
    }

    public void releaseParkingSpot(ParkingSpot parkingSpot) {
        lock.lock();
        try {
            parkingSpot.setOccupied(false);
        } finally {
            lock.unlock();
        }
    }
}

**`ParkingListener`:** that react to park and unpark events:

In [ ]:
public interface ParkingListener {
    void onPark(ParkingTicket parkingTicket);
    void onUnpark(ParkingTicket parkingTicket);
}

public class ParkingDisplay implements ParkingListener {
    @Override
    public void onPark(ParkingTicket parkingTicket) {
        var message = """
                Vehicle with registration number : %s
                    Parked at time : %s
                    On spot %s on floor : %s
                """.formatted(parkingTicket.getVehicle().getRegistrationNumber(),
                parkingTicket.getStartTime(),
                parkingTicket.getParkingSpot().getId(),
                parkingTicket.getParkingSpot().getFloor()
        );

        System.out.println(message);
    }

    @Override
    public void onUnpark(ParkingTicket parkingTicket) {
        var message = """
                Vehicle with registration number : %s
                    Parked at time : %s
                    Unparked at time : %s
                    On spot %s on floor : %s
                    Total amount due : %s
                """.formatted(parkingTicket.getVehicle().getRegistrationNumber(),
                parkingTicket.getStartTime(),
                parkingTicket.getEndTime(),
                parkingTicket.getParkingSpot().getId(),
                parkingTicket.getParkingSpot().getFloor(),
                parkingTicket.getFees()
        );

        System.out.println(message);
    }
}

**`FeesStrategy`:** that calculates parking fees using different strategies:

In [ ]:
public interface FeeStrategy {
    BigDecimal calculate(ParkingTicket ticket);
}

public class FlatFeeStrategy implements FeeStrategy{
    private EnumMap<ParkingSpotSize, BigDecimal> fees;

    public FlatFeeStrategy() {
        fees = new EnumMap<>(ParkingSpotSize.class);
        fees.put(ParkingSpotSize.SMALL, BigDecimal.valueOf(10));
        fees.put(ParkingSpotSize.MEDIUM, BigDecimal.valueOf(15));
        fees.put(ParkingSpotSize.LARGE, BigDecimal.valueOf(20));
    }

    @Override
    public BigDecimal calculate(ParkingTicket ticket) {
        return fees.get(ticket.getParkingSpot().getSize());
    }
}

public class HourlyFeesStrategy implements FeeStrategy {
    private EnumMap<ParkingSpotSize, BigDecimal> rate;

    public HourlyFeesStrategy() {
        rate = new EnumMap<>(ParkingSpotSize.class);
        rate.put(ParkingSpotSize.SMALL, BigDecimal.valueOf(5));
        rate.put(ParkingSpotSize.MEDIUM, BigDecimal.valueOf(8));
        rate.put(ParkingSpotSize.LARGE, BigDecimal.valueOf(10));
    }

    @Override
    public BigDecimal calculate(ParkingTicket ticket) {
        Duration duration = Duration.between(ticket.getStartTime(), ticket.getEndTime());
        long minutes = duration.toMinutes();
        long hours = Math.max(1, (long) Math.ceil(minutes / 60.0));
        return rate.get(ticket.getParkingSpot().getSize()).multiply(BigDecimal.valueOf(hours));
    }
}

**`ParkingLot`:** a singleton that acts as the conduit for parking and unparking vehicles.

In [ ]:
@Getter
public class ParkingLot {
    private static volatile ParkingLot INSTANCE;

    private final Map<Integer, ParkingFloor> floorMap;
    private volatile FeeStrategy feeStrategy;
    private final List<ParkingListener> parkingListeners;

    private ParkingLot(List<ParkingFloor> floors,
                       FeeStrategy feeStrategy,
                       List<ParkingListener> parkingListeners) {
        this.floorMap = floors.stream()
                .collect(Collectors.toMap(ParkingFloor::getFloor, f -> f));
        this.feeStrategy = feeStrategy;
        this.parkingListeners = new ArrayList<>(parkingListeners); // Defensive copy
    }

    public static ParkingLot getInstance(List<ParkingFloor> floors,
                                         FeeStrategy feeStrategy,
                                         List<ParkingListener> parkingListeners) {
        if (INSTANCE == null) {
            synchronized (ParkingLot.class) {
                if (INSTANCE == null) {
                    INSTANCE = new ParkingLot(floors, feeStrategy, parkingListeners);
                }
            }
        }
        return INSTANCE;
    }

    public static ParkingLot getInstance() {
        if (INSTANCE == null) {
            throw new IllegalStateException(
                    "ParkingLot not initialized. Call getInstance(floors, strategy, listeners) first.");
        }
        return INSTANCE;
    }

    public ParkingTicket park(Vehicle vehicle) throws NoSpotAvailableException {
        for (ParkingFloor floor : floorMap.values()) {
            ParkingSpot spot = floor.getParkingSpot(vehicle.getVehicleType().compatibleSpots());
            if (spot != null) {
                ParkingTicket ticket = new ParkingTicket();
                ticket.setParkingSpot(spot);
                ticket.setVehicle(vehicle);
                parkingListeners.forEach(p -> p.onPark(ticket));
                return ticket;
            }
        }

        throw new NoSpotAvailableException(
                "No spot available for vehicle type " + vehicle.getVehicleType());
    }

    public void unpark(ParkingTicket ticket) {
        if (ticket.getStatus() != TicketStatus.ACTIVE) {
            throw new IllegalStateException(
                    "Ticket " + ticket.getId() + " is already " + ticket.getStatus());
        }

        ticket.setEndTime(LocalDateTime.now());
        ticket.setFees(feeStrategy.calculate(ticket));
        ticket.setStatus(TicketStatus.PAID);

        parkingListeners.forEach(p -> p.onUnpark(ticket));

        ParkingSpot spot = ticket.getParkingSpot();
        ParkingFloor floor = floorMap.get(spot.getFloor());
        floor.releaseParkingSpot(spot);
    }

    public void setFeeStrategy(FeeStrategy feeStrategy) {
        this.feeStrategy = feeStrategy; // volatile write — safe without locking
    }
}

public class NoSpotAvailableException extends Exception {
    public NoSpotAvailableException(String message) {
        super(message);
    }
}